# 🧪 实验 01：`H3-Base-FL2VA` 官方推理黄金组合 (42.03 GB) —— 全链路测速（云端 $\rightarrow$ 本地磁盘 $\rightarrow$ 内存 RAM $\rightarrow$ 显存 VRAM）

---

## 1. Colab 全系可选 GPU 规格表（基于 `colab-cli` 源码核实）

| `colab-cli` 参数 | 对应 GPU 硬件型号 | 架构 | 显存 (VRAM) | High-RAM 系统内存 | 部署 `H3-Base-FL2VA` (42.03 GB) 评价 |
| :--- | :--- | :--- | :---: | :---: | :--- |
| **`G4`** (`--gpu G4 --high-mem`) | **NVIDIA RTX PRO 6000 Blackwell Server Edition** | **Blackwell** | **96 GB GDDR7** | ~83.5 GB+ | 👑 **天花板首选！** 96GB 显存可将 42.03GB 全套权重一次性全塞进显存常驻（All-in-VRAM），零 Offload 开销，且支持原生 Blackwell FP4/INT8/FP8 硬件加速。 |
| **`H100`** (`--gpu H100 --high-mem`) | **NVIDIA H100** | Hopper | **80 GB HBM3** | ~83.5 GB+ | 🚀 **极速档**。80GB 显存同样支持 42.03GB 全套常驻显存。 |
| **`A100`** (`--gpu A100 --high-mem`) | **NVIDIA A100** | Ampere | **40 GB / 80 GB** | ~83.5 GB | ✅ **主力稳定档**。40GB 版通过内存 Offload 轮流切换模块；80GB 版可全常驻显存。 |
| **`L4`** (`--gpu L4`) | **NVIDIA L4** | Ada Lovelace | **24 GB GDDR6** | ~53.0 GB | 💡 **高性价比档**。53GB 系统内存可装下 42.03GB 全套模型，配合流式 Offload 运行。 |
| **`T4`** (`--gpu T4 --high-mem`) | **NVIDIA Tesla T4** | Turing | **15 GB GDDR6** | ~12.7 / 51 GB | ❌ **不推荐**。15GB 显存装不下 21GB 主干。 |

---

## 2. 本实验测试的 3 条端到端真实链路

1. **链路 B1（Drive 拷贝流）**：`Google Drive` $\rightarrow$ ① 拷贝至 `/content` 本地 SSD $\rightarrow$ ② `safetensors.load_file` 至系统内存 `RAM` $\rightarrow$ ③ `.to("cuda")` 搬运至 GPU 显存 `VRAM`
2. **链路 B2（Drive 零拷贝直读流）**：`Google Drive` (`/content/drive`) $\rightarrow$ ① 零拷贝 $\rightarrow$ ② 直接从云盘挂载路径 `load_file` 至系统内存 `RAM` $\rightarrow$ ③ `.to("cuda")` 搬运至 GPU 显存 `VRAM`
3. **链路 A（HF 直连下载流）**：`Hugging Face (hf_transfer)` $\rightarrow$ ① 下载至 `/content` 本地 SSD $\rightarrow$ ② `safetensors.load_file` 至系统内存 `RAM` $\rightarrow$ ③ `.to("cuda")` 搬运至 GPU 显存 `VRAM`

In [ ]:
# @title Step 1: 挂载 Google Drive、自动体检【① 磁盘 / ② 内存 / ③ 显存】并加载 42.03 GB 黄金清单
!pip install -q -U huggingface_hub hf_transfer safetensors torch psutil

import gc, os, shutil, subprocess, time, psutil, torch
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from pathlib import Path
from google.colab import drive
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

drive.mount('/content/drive')

REPO_ID = "Comfy-Org/MiniMax-H3"
GDRIVE_MODEL_ROOT = Path("/content/drive/MyDrive/models/Comfy-MiniMax-H3-FL2VA")
LOCAL_DEST = Path("/content/Comfy-MiniMax-H3-FL2VA")

GOLD_BUNDLE_FILES = [
    {
        "role": "FL2VA 生成主干 (Pruned INT8 ConvRot)",
        "path": "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors",
        "size": 20970379616,  # 20.97 GB
    },
    {
        "role": "文本编码器 (Qwen3-VL-32B NVFP4 AWQ)",
        "path": "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
        "size": 15687142551,  # 15.69 GB
    },
    {
        "role": "视频解码器 (Video VAE INT8 ConvRot)",
        "path": "vae/minimax_h3_video_vae_int8_convrot.safetensors",
        "size": 2811065184,   # 2.81 GB
    },
    {
        "role": "8步加速 LoRA (FL2V Turbo 8-Step)",
        "path": "loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors",
        "size": 1956193000,   # 1.96 GB
    },
    {
        "role": "音频解码器 (Audio VAE FP32)",
        "path": "vae/minimax_h3_audio_vae_fp32.safetensors",
        "size": 605254808,    # 0.61 GB
    },
]

def free_space_gib(path=Path("/content")) -> float:
    return shutil.disk_usage(path).free / (1024 ** 3)

def total_space_gib(path=Path("/content")) -> float:
    return shutil.disk_usage(path).total / (1024 ** 3)

def purge_drivefs_and_page_cache():
    """清理 Linux 内核 Page Cache (切勿在挂载状态下 rmtree DriveFS/content_cache，否则会导致 FUSE 挂载点崩溃)"""
    subprocess.run(["sync"], check=False)
    if Path("/proc/sys/vm/drop_caches").exists():
        with open("/proc/sys/vm/drop_caches", "w") as f:
            f.write("3\n")

def measure_disk_to_ram_and_vram(file_path: Path):
    """测量单个 safetensors 文件: ② Disk -> System RAM 耗时 与 ③ System RAM -> GPU VRAM 耗时"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    t0 = time.perf_counter()
    cpu_tensors = load_file(str(file_path), device="cpu")
    ram_sec = time.perf_counter() - t0

    vram_sec = 0.0
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        gpu_tensors = {k: v.to("cuda", non_blocking=False) for k, v in cpu_tensors.items()}
        torch.cuda.synchronize()
        vram_sec = time.perf_counter() - t1
        del gpu_tensors
        torch.cuda.empty_cache()

    del cpu_tensors
    gc.collect()
    return ram_sec, vram_sec

ram_total_gib = psutil.virtual_memory().total / (1024 ** 3)
disk_total_gib = total_space_gib()
disk_free_gib = free_space_gib()
total_gb = sum(f["size"] for f in GOLD_BUNDLE_FILES) / 1e9
total_gib = sum(f["size"] for f in GOLD_BUNDLE_FILES) / (1024 ** 3)

print("=" * 84)
print("🔍 当前 Colab 虚拟机【① 磁盘 -> ② 内存 -> ③ 显存】三关硬件体检：")
print(f"  ① 本地磁盘 (Disk /content) : 总计 {disk_total_gib:.1f} GiB | 当前可用 {disk_free_gib:.1f} GiB")
print(f"  ② 系统内存 (System RAM)    : 总计 {ram_total_gib:.1f} GiB")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gib = props.total_memory / (1024 ** 3)
    print(f"  ③ GPU 显存 (VRAM)          : {props.name} | 总计 {vram_gib:.1f} GiB")
else:
    print("  ③ GPU 显存 (VRAM)          : 当前为 CPU 模式 (若需测阶段 ③ RAM->VRAM，请切换至 G4 / A100 / L4 GPU)")
print(f"  📦 黄金推理权重包总大小     : {total_gb:.2f} GB ({total_gib:.2f} GiB, 共 5 个文件)")
print("=" * 84)

In [ ]:
# @title Step 2: 将 5 个黄金权重文件 (42.03 GB) 持久化存入你的 Google Drive（支持断点续传，可在免费 CPU 下先跑完）
from huggingface_hub import hf_hub_download
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

staging_dir = Path("/content/_fl2va_staging")
staging_dir.mkdir(parents=True, exist_ok=True)
GDRIVE_MODEL_ROOT.mkdir(parents=True, exist_ok=True)

total_bytes = sum(f["size"] for f in GOLD_BUNDLE_FILES)
done_bytes = 0
t_start = time.perf_counter()

for idx, item in enumerate(GOLD_BUNDLE_FILES, 1):
    rel_path = item["path"]
    expected_size = item["size"]
    gdrive_file = GDRIVE_MODEL_ROOT / rel_path

    if gdrive_file.exists() and gdrive_file.stat().st_size == expected_size:
        done_bytes += expected_size
        print(f"  [{idx}/5] ⏭️ 已存在跳过: {item['role']} ({expected_size/1e9:.2f} GB)")
        continue

    print(f"  [{idx}/5] ⬇️ 正在下载并写入 Google Drive: {item['role']} ({expected_size/1e9:.2f} GB) | 本地剩余: {free_space_gib():.1f} GiB")
    staged_file = Path(hf_hub_download(repo_id=REPO_ID, filename=rel_path, local_dir=str(staging_dir), force_download=True))
    shutil.rmtree(staging_dir / ".cache", ignore_errors=True)

    gdrive_file.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(staged_file, gdrive_file)
    staged_file.unlink(missing_ok=True)
    purge_drivefs_and_page_cache()

    done_bytes += expected_size
    pct = (done_bytes / total_bytes) * 100
    elapsed_min = (time.perf_counter() - t_start) / 60.0
    print(f"        ✅ 已存云盘 ({pct:.1f}% - {done_bytes/1e9:.2f}/{total_bytes/1e9:.2f} GB, 累计 {elapsed_min:.1f} 分钟)")

shutil.rmtree(staging_dir, ignore_errors=True)
purge_drivefs_and_page_cache()
print(f"\n🎉 黄金组合 5 个权重文件 (42.03 GB) 已全部永久保存在 Google Drive: {GDRIVE_MODEL_ROOT}")

In [ ]:
# @title Step 3: 执行 42.03 GB 全链路对比测试（① 落盘 -> ② Disk入RAM -> ③ RAM入VRAM）
from huggingface_hub import hf_hub_download
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

results = []
total_bytes = sum(f["size"] for f in GOLD_BUNDLE_FILES)

# ==============================================================================
# 链路 B1: Google Drive -> 拷贝至 /content 本地盘 -> 加载至 RAM -> 搬运至 VRAM
# ==============================================================================
print("=" * 88)
print("▶️ [链路 B1] Google Drive -> ① 拷贝至 /content -> ② 加载至 RAM -> ③ 搬运至 VRAM")
print("=" * 88)
shutil.rmtree(LOCAL_DEST, ignore_errors=True)
LOCAL_DEST.mkdir(parents=True, exist_ok=True)
purge_drivefs_and_page_cache()

b1_tx, b1_ram, b1_vram = 0.0, 0.0, 0.0
for idx, item in enumerate(GOLD_BUNDLE_FILES, 1):
    rel_path, fsize = item["path"], item["size"]
    src_file = GDRIVE_MODEL_ROOT / rel_path
    dst_file = LOCAL_DEST / rel_path
    dst_file.parent.mkdir(parents=True, exist_ok=True)

    t0 = time.perf_counter()
    shutil.copy2(src_file, dst_file)
    tx_s = time.perf_counter() - t0

    purge_drivefs_and_page_cache()
    ram_s, vram_s = measure_disk_to_ram_and_vram(dst_file)

    b1_tx += tx_s
    b1_ram += ram_s
    b1_vram += vram_s
    mbs = (fsize / (1024 ** 2)) / tx_s
    print(f"  [B1 {idx}/5] {item['role']} ({fsize/1e9:.2f} GB) | ①拷贝:{tx_s:.1f}s ({mbs:.0f}MB/s) | ②入RAM:{ram_s:.1f}s | ③入VRAM:{vram_s:.1f}s")

shutil.rmtree(LOCAL_DEST, ignore_errors=True)
purge_drivefs_and_page_cache()
results.append({
    "name": "链路 B1: Google Drive -> 拷贝至 /content -> RAM -> VRAM",
    "tx_s": b1_tx,
    "tx_mbs": (total_bytes / (1024 ** 2)) / b1_tx,
    "ram_s": b1_ram,
    "vram_s": b1_vram,
    "total_s": b1_tx + b1_ram + b1_vram,
})

# ==============================================================================
# 链路 B2: Google Drive 原地零拷贝 -> 直接从 /content/drive 加载至 RAM -> 搬运至 VRAM
# ==============================================================================
print("\n" + "=" * 88)
print("▶️ [链路 B2] Google Drive 原地零拷贝 -> ② 直接从 /content/drive 加载至 RAM -> ③ 搬运至 VRAM")
print("=" * 88)
purge_drivefs_and_page_cache()

b2_ram, b2_vram = 0.0, 0.0
for idx, item in enumerate(GOLD_BUNDLE_FILES, 1):
    rel_path, fsize = item["path"], item["size"]
    src_file = GDRIVE_MODEL_ROOT / rel_path

    purge_drivefs_and_page_cache()
    ram_s, vram_s = measure_disk_to_ram_and_vram(src_file)

    b2_ram += ram_s
    b2_vram += vram_s
    mbs = (fsize / (1024 ** 2)) / ram_s
    print(f"  [B2 {idx}/5] {item['role']} ({fsize/1e9:.2f} GB) | ①零拷贝 | ②直读入RAM:{ram_s:.1f}s ({mbs:.0f}MB/s) | ③入VRAM:{vram_s:.1f}s")
    purge_drivefs_and_page_cache()

results.append({
    "name": "链路 B2: Google Drive 原地零拷贝 -> RAM -> VRAM",
    "tx_s": 0.0,
    "tx_mbs": (total_bytes / (1024 ** 2)) / b2_ram,
    "ram_s": b2_ram,
    "vram_s": b2_vram,
    "total_s": b2_ram + b2_vram,
})

# ==============================================================================
# 链路 A: Hugging Face (hf_transfer) -> 下载至 /content 本地盘 -> 加载至 RAM -> 搬运至 VRAM
# ==============================================================================
print("\n" + "=" * 88)
print("▶️ [链路 A] Hugging Face (hf_transfer) -> ① 下载至 /content -> ② 加载至 RAM -> ③ 搬运至 VRAM")
print("=" * 88)
LOCAL_DEST.mkdir(parents=True, exist_ok=True)
purge_drivefs_and_page_cache()

a_tx, a_ram, a_vram = 0.0, 0.0, 0.0
for idx, item in enumerate(GOLD_BUNDLE_FILES, 1):
    rel_path, fsize = item["path"], item["size"]

    t0 = time.perf_counter()
    dst_file = Path(hf_hub_download(repo_id=REPO_ID, filename=rel_path, local_dir=str(LOCAL_DEST), force_download=True))
    tx_s = time.perf_counter() - t0
    shutil.rmtree(LOCAL_DEST / ".cache", ignore_errors=True)

    purge_drivefs_and_page_cache()
    ram_s, vram_s = measure_disk_to_ram_and_vram(dst_file)

    a_tx += tx_s
    a_ram += ram_s
    a_vram += vram_s
    mbs = (fsize / (1024 ** 2)) / tx_s
    print(f"  [A  {idx}/5] {item['role']} ({fsize/1e9:.2f} GB) | ①HF下载:{tx_s:.1f}s ({mbs:.0f}MB/s) | ②入RAM:{ram_s:.1f}s | ③入VRAM:{vram_s:.1f}s")

results.append({
    "name": "链路 A : Hugging Face (hf_transfer) -> /content -> RAM -> VRAM",
    "tx_s": a_tx,
    "tx_mbs": (total_bytes / (1024 ** 2)) / a_tx,
    "ram_s": a_ram,
    "vram_s": a_vram,
    "total_s": a_tx + a_ram + a_vram,
})

# ==============================================================================
# 输出最终对比报告
# ==============================================================================
best = min(results, key=lambda r: r["total_s"])
print("\n" + "=" * 105)
print("📊 实验 01 全链路实测报告：FL2VA 黄金组合 (42.03 GB) 【云端 -> 本地盘 -> 内存 RAM -> 显存 VRAM】")
print("=" * 105)
print("| 端到端链路方案 | ① 落盘耗时 (吞吐率) | ② Disk -> RAM 耗时 | ③ RAM -> VRAM 耗时 | **端到端总就绪耗时** |")
print("| :--- | :---: | :---: | :---: | :---: |")
for r in results:
    crown = " 🏆" if r is best else ""
    tx_str = "0.0 s (零拷贝)" if r["tx_s"] == 0.0 else f"{r['tx_s']:.1f} s ({r['tx_mbs']:.0f} MB/s)"
    print(f"| **{r['name']}{crown}** | {tx_str} | {r['ram_s']:.1f} s | {r['vram_s']:.1f} s | **{r['total_s']:.1f} s ({r['total_s']/60:.2f} min)** |")
print("=" * 105)